In [0]:
%sql
USE CATALOG workspace;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS project_22a;

In [0]:
%sql
USE SCHEMA project_22a;

In [0]:
%sql
SELECT current_catalog(),current_schema();

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS project_22a_data;

In [0]:
%sql
SELECT * 
FROM read_files(
    '/Volumes/workspace/project_22a/project_22a_data/raw_customers.json',
    FORMAT => 'json'
);

In [0]:
%sql
CREATE OR REPLACE TABLE STG_CUSTOMERS 
(
    CUSTOMER_ID STRING,
    NAME STRING,
    ADDRESS STRING,
    TIER STRING,
    PHONE STRING,
    EFFECTIVE_DATE DATE
);

In [0]:
%sql
INSERT INTO STG_CUSTOMERS(CUSTOMER_ID,
                          NAME,
                          ADDRESS,
                          TIER,
                          PHONE,
                          EFFECTIVE_DATE)
SELECT  CUSTOMER_ID,
        NAME,
        ADDRESS,
        TIER,
        PHONE,
        EFFECTIVE_DATE 
FROM read_files(
    '/Volumes/workspace/project_22a/project_22a_data/raw_customers.json',
    FORMAT => 'json'
);

In [0]:
%sql
SELECT * 
FROM STG_CUSTOMERS;

In [0]:
%sql
SELECT  'STG_CUST_UPD' AS STAGING_TABLE,
        COUNT(*) AS LOADED_ROWS
FROM STG_CUSTOMERS; 

In [0]:
%sql
CREATE TABLE DIM_CUSTOMER_UPDS
(
    CUSTOMER_SK STRING,
    CUSTOMER_ID STRING,
    NAME STRING,
    ADDRESS STRING,
    TIER STRING,
    PHONE STRING,
    START_DATE DATE,
    END_DATE DATE,
    IS_CURRENT BOOLEAN
);

In [0]:
%sql
INSERT INTO DIM_CUSTOMER_UPDS(CUSTOMER_SK,
                              CUSTOMER_ID,
                              NAME,
                              ADDRESS,
                              TIER,
                              PHONE,
                              START_DATE,
                              END_DATE,
                              IS_CURRENT)
SELECT  CUSTOMER_SK,
        CUSTOMER_ID,
        NAME,
        ADDRESS,
        TIER,
        PHONE,
        START_DATE,
        END_DATE,
        IS_CURRENT
FROM read_files(
    '/Volumes/workspace/project_22a/project_22a_data/dim_customer_scd2_active.json',
    FORMAT => 'json'
);

In [0]:
%sql
SELECT *
FROM DIM_CUSTOMER_UPDS;

In [0]:
%sql
SELECT  CUSTOMER_ID,
        SHA2(CONCAT(COALESCE(ADDRESS,''),'|',(COALESCE(TIER,''))),256) AS RECORD_HASH
FROM STG_CUSTOMERS;

In [0]:
%sql
CREATE TEMPORARY VIEW TASK3_RESULT_VW AS 
WITH STG_HASH AS
(
    SELECT  CUSTOMER_ID,
            SHA2(CONCAT(COALESCE(ADDRESS,''),'|',(COALESCE(TIER,''))),256) AS RECORD_HASH
    FROM STG_CUSTOMERS
), DIM_HASH AS
(
    SELECT  CUSTOMER_ID,
            SHA2(CONCAT(COALESCE(ADDRESS,''),'|',(COALESCE(TIER,''))),256) AS RECORD_HASH
    FROM DIM_CUSTOMER_UPDS
),UPD_CUST AS
(
SELECT  STG.CUSTOMER_ID,
        CASE WHEN D.CUSTOMER_ID IS NULL 
        THEN 'NO(NEW)'
        WHEN SH.RECORD_HASH = DH.RECORD_HASH AND (STG.PHONE <> D.PHONE)
        THEN 'YES'
        WHEN SH.RECORD_HASH <> DH.RECORD_HASH
        THEN 'YES'
        ELSE 'NO'
        END AS CHANGE_DETECTED,
        CASE WHEN D.CUSTOMER_ID IS NULL 
        THEN 'NO'
        WHEN SH.RECORD_HASH = DH.RECORD_HASH AND (STG.PHONE <> D.PHONE)
        THEN 'YES(PHONE ONLY)'
        WHEN SH.RECORD_HASH <> DH.RECORD_HASH 
        THEN 'NO'
        ELSE 'NO'
        END AS SCD1_ONLY_UPDATE,
        CASE WHEN D.CUSTOMER_ID IS NULL 
        THEN 'INSERT_NEW_CUSTOMER'
        WHEN SH.RECORD_HASH = DH.RECORD_HASH AND (STG.PHONE <> D.PHONE)
        THEN 'UPDATE_SCD1_INLINE'
        WHEN SH.RECORD_HASH <> DH.RECORD_HASH 
        THEN 'EXPIRE_AND_INSERT_SCD2'
        ELSE 'NO_ACTION'
        END AS ACTION_REQUIRED
FROM STG_CUSTOMERS STG
LEFT JOIN DIM_CUSTOMER_UPDS D
ON STG.CUSTOMER_ID = D.CUSTOMER_ID
LEFT JOIN STG_HASH SH
ON SH.CUSTOMER_ID = STG.CUSTOMER_ID
LEFT JOIN DIM_HASH DH
ON DH.CUSTOMER_ID = STG.CUSTOMER_ID
)SELECT * FROM UPD_CUST
ORDER BY CUSTOMER_ID;


In [0]:
%sql
SELECT * 
FROM TASK3_RESULT_VW;

In [0]:
%sql

MERGE INTO DIM_CUSTOMER_UPDS TAR
USING (SELECT STG.CUSTOMER_ID,
              STG.EFFECTIVE_DATE
      FROM STG_CUSTOMERS STG
      JOIN TASK3_RESULT_VW T  
      ON STG.CUSTOMER_ID = T.CUSTOMER_ID
      WHERE T.ACTION_REQUIRED = 'EXPIRE_AND_INSERT_SCD2') SOU
ON TAR.CUSTOMER_ID = SOU.CUSTOMER_ID AND TAR.IS_CURRENT = TRUE
WHEN MATCHED THEN
    UPDATE 
        SET TAR.END_DATE = DATE_SUB(SOU.EFFECTIVE_DATE,1),
            TAR.IS_CURRENT = FALSE;

In [0]:
%sql
INSERT INTO DIM_CUSTOMER_UPDS(CUSTOMER_SK,
                              CUSTOMER_ID,
                              NAME,
                              ADDRESS,
                              TIER,
                              PHONE,
                              START_DATE,
                              END_DATE,
                              IS_CURRENT)
SELECT  CONCAT('SK-',(SELECT MAX(CAST(RIGHT(CUSTOMER_SK,4) AS INT)) FROM DIM_CUSTOMER_UPDS) + ROW_NUMBER() OVER (ORDER BY STG.CUSTOMER_ID)) AS CUSTOMER_SK,
        STG.CUSTOMER_ID,
        STG.NAME,
        STG.ADDRESS,
        STG.TIER,
        STG.PHONE,
        STG.EFFECTIVE_DATE,
        TO_DATE('9999-12-31') AS END_DATE,
        TRUE AS IS_CURRENT
FROM STG_CUSTOMERS STG
JOIN TASK3_RESULT_VW VW
ON STG.CUSTOMER_ID = VW.CUSTOMER_ID
WHERE VW.ACTION_REQUIRED IN ('INSERT_NEW_CUSTOMER','EXPIRE_AND_INSERT_SCD2');

In [0]:
%sql
SELECT * FROM DIM_CUSTOMER_UPDS
ORDER BY CUSTOMER_ID,CUSTOMER_SK;

In [0]:
%sql
MERGE INTO DIM_CUSTOMER_UPDS T
USING (SELECT STG.CUSTOMER_ID,
              STG.PHONE 
       FROM STG_CUSTOMERS STG 
       JOIN TASK3_RESULT_VW VW 
       ON STG.CUSTOMER_ID = VW.CUSTOMER_ID 
       WHERE ACTION_REQUIRED = 'UPDATE_SCD1_INLINE') S
ON T.CUSTOMER_ID = S.CUSTOMER_ID AND T.IS_CURRENT = TRUE
WHEN MATCHED THEN
    UPDATE 
        SET T.PHONE = S.PHONE;


In [0]:
%sql
WITH OVERLAPS AS
(
SELECT
A.CUSTOMER_ID
FROM DIM_CUSTOMER_UPDS A
JOIN DIM_CUSTOMER_UPDS B
ON A.CUSTOMER_ID = B.CUSTOMER_ID
AND A.CUSTOMER_SK <> B.CUSTOMER_SK
AND A.START_DATE < B.END_DATE
AND B.START_DATE < A.END_DATE
)
SELECT
'TEMPORAL_RANGE_CHECK' AS CHECK_NAME,
COUNT(*) AS OVERLAPS_FOUND,
CASE
WHEN COUNT(*) = 0 THEN 'PASSED'
ELSE 'FAILED'
END AS STATUS
FROM OVERLAPS;

In [0]:
%sql
DESCRIBE HISTORY DIM_CUSTOMER_UPDS;

In [0]:
%sql
SELECT  CUSTOMER_ID,
        ADDRESS,
        TIER,
        START_DATE,
        IS_CURRENT
FROM DIM_CUSTOMER_UPDS VERSION AS OF 3
WHERE CUSTOMER_ID ='CUST-101';

In [0]:
%sql
OPTIMIZE DIM_CUSTOMER_UPDS
ZORDER BY (CUSTOMER_SK);

In [0]:
%sql
DESCRIBE HISTORY DIM_CUSTOMER_UPDS;

In [0]:
%sql
SELECT  'DIM_CUSTOMER_SCD2' AS TARGET_TABLE,
        'OPTIMIZED' AS CLUSTERING_STATUS;